In [1]:
import urllib.request
import numpy as np
import re
import collections
import random

In [2]:
url = "https://www.gutenberg.org/files/11/11-0.txt"
response = urllib.request.urlopen(url)
raw_text = response.read().decode('utf-8')
text = re.sub(r'[^a-zA-Z\s]', '', raw_text).lower()
tokens = text.split()

print(f"Total words in book: {len(tokens)}")

Total words in book: 26476


In [7]:
word_counts = collections.Counter(tokens)
min_count = 5
vocab = [word for word, count in word_counts.items() if count >= min_count]
vocab = sorted(vocab)
word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for i, word in enumerate(vocab)}
vocab_size = len(vocab)
corpus_ids = [word_to_id[word] for word in tokens if word in word_to_id]

print(f"Original unique words: {len(word_counts)}")
print(f"Vocabulary size after filtering: {vocab_size}")
print(f"Total valid words in corpus: {len(corpus_ids)}")

Original unique words: 2762
Vocabulary size after filtering: 684
Total valid words in corpus: 23078


In [11]:
def generate_training_data(corpus_ids, window_size=2):
    """
    Yields one (center_id, context_id) pair at a time to save memory.
    """
    for i, center_id in enumerate(corpus_ids):
        start = max(0, i - window_size)
        end = min(len(corpus_ids), i + window_size + 1)

        for j in range(start, end):
            if i != j:
                context_id = corpus_ids[j]
                yield center_id, context_id

In [12]:
class Word2VecNumPy:
    def __init__(self, vocab_size, embed_size=10, lr=0.05):
        self.vocab_size = vocab_size
        self.embed_size = embed_size
        self.lr = lr

        self.W_in = np.random.uniform(-0.5, 0.5, (vocab_size, embed_size))
        self.W_out = np.random.uniform(-0.5, 0.5, (vocab_size, embed_size))

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -10, 10)))

    def train_step(self, center_id, context_id, negative_samples):

        v_c = self.W_in[center_id]
        u_o = self.W_out[context_id]

        z_pos = np.dot(v_c, u_o)
        prob_pos = self.sigmoid(z_pos)

        loss = -np.log(prob_pos + 1e-9)

        error_pos = prob_pos - 1

        grad_v_c = error_pos * u_o
        grad_u_o = error_pos * v_c

        self.W_out[context_id] -= self.lr * grad_u_o

        # --- Negative Sampling ---
        for neg_id in negative_samples:
            u_n = self.W_out[neg_id]
            z_neg = np.dot(v_c, u_n)
            prob_neg = self.sigmoid(z_neg)

            loss -= np.log(1 - prob_neg + 1e-9)
            error_neg = prob_neg - 0

            grad_v_c += error_neg * u_n
            self.W_out[neg_id] -= self.lr * (error_neg * v_c)
        self.W_in[center_id] -= self.lr * grad_v_c

        return loss

In [13]:
embed_size = 300
model = Word2VecNumPy(vocab_size, embed_size=embed_size, lr=0.025)

epochs = 3
num_neg_samples = 5
window_size = 2

for epoch in range(epochs):
    total_loss = 0
    pair_count = 0
    data_generator = generate_training_data(corpus_ids, window_size)

    for center_id, context_id in data_generator:
        neg_samples = []
        while len(neg_samples) < num_neg_samples:
            neg_id = random.randint(0, vocab_size - 1)
            if neg_id != context_id:
                neg_samples.append(neg_id)

        loss = model.train_step(center_id, context_id, neg_samples)
        total_loss += loss
        pair_count += 1

        if pair_count % 50000 == 0:
            print(f"Epoch {epoch+1} | Pairs processed: {pair_count} | Avg Loss: {total_loss/pair_count:.4f}")

    print(f"--- End of Epoch {epoch+1} ---")

Epoch 1 | Pairs processed: 50000 | Avg Loss: 3.0151
--- End of Epoch 1 ---
Epoch 2 | Pairs processed: 50000 | Avg Loss: 1.8086
--- End of Epoch 2 ---
Epoch 3 | Pairs processed: 50000 | Avg Loss: 1.6513
--- End of Epoch 3 ---


In [9]:
def get_most_similar(target_word, top_k=5):
    if target_word not in word_to_id:
        return f"'{target_word}' is not in the vocabulary."

    target_id = word_to_id[target_word]
    target_vec = model.W_in[target_id]

    dot_products = np.dot(model.W_in, target_vec)
    norms = np.linalg.norm(model.W_in, axis=1)
    target_norm = np.linalg.norm(target_vec)
    similarities = dot_products / (norms * target_norm)
    top_indices = np.argsort(similarities)[::-1]

    print(f"Words most similar to '{target_word}':")
    results_printed = 0

    for idx in top_indices:
        if idx != target_id:
            word = id_to_word[idx]
            score = similarities[idx]
            print(f" - {word} (Score: {score:.4f})")
            results_printed += 1

            if results_printed == top_k:
                break

get_most_similar('alice', top_k=5)
print("-" * 30)
get_most_similar('rabbit', top_k=5)
print("-" * 30)
get_most_similar('said', top_k=5)

Words most similar to 'alice':
 - said (Score: 0.3445)
 - but (Score: 0.3411)
 - it (Score: 0.3322)
 - moment (Score: 0.3227)
 - she (Score: 0.3080)
------------------------------
Words most similar to 'rabbit':
 - lobster (Score: 0.2904)
 - white (Score: 0.2872)
 - hall (Score: 0.2836)
 - good (Score: 0.2803)
 - call (Score: 0.2748)
------------------------------
Words most similar to 'said':
 - thing (Score: 0.3449)
 - alice (Score: 0.3445)
 - went (Score: 0.3277)
 - it (Score: 0.3180)
 - thought (Score: 0.3021)
